In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [2]:
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")




Querying owner: HTOC Org

Retrieved 659 unique observed indicators.


In [3]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,dnsActive,whoisActive,source,url,text,sha256,address,Subject,md5,indicator
0,6755399459033718,2025-06-16T17:42:07Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-17T11:59:05Z,3.0,80.0,RITM0589984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.70.100.68
1,5629499555060692,2025-06-16T17:42:26Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-17T11:58:47Z,3.0,80.0,RITM0589984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.141.215.88
2,5629499555060694,2025-06-16T17:42:28Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-17T11:58:32Z,3.0,80.0,RITM0589984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.42.116.215
3,5629499555060667,2025-06-16T17:42:12Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-17T11:58:32Z,3.0,80.0,RITM0589984,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.42.116.178
4,6755399458556994,2025-06-11T14:51:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-17T11:58:28Z,3.0,80.0,RITM0588707 / TASK0886419,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,161.35.53.231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9923,5629499537015559,2025-04-23T15:01:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-12T11:24:51Z,5.0,93.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,103.108.231.51
9924,5629499537015520,2025-04-23T15:01:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-12T11:24:51Z,5.0,93.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,103.216.220.35
9926,5629499537015453,2025-04-23T15:01:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-12T11:24:51Z,5.0,93.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,129.227.46.131
9950,5265145,2025-01-23T19:51:13Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-12T11:24:51Z,3.0,66.0,CISA conducted a hunt on IoC's obtained from o...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.227.106.112


In [3]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=5)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250617.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250616.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250615.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250614.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250613.csv']
Loaded data from 5 files.


In [4]:
import pandas as pd
from datetime import timedelta, date

# Define cutoff time in UTC
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# Initialize an empty DataFrame to store filtered tags
filtered_tags = pd.DataFrame()

# Loop through each row in observed_src
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    
    if isinstance(tags_data, list):
        # Normalize the tags data for the current row
        tags = pd.json_normalize(tags_data)

        # Ensure 'name' column is of string type
        tags['name'] = tags['name'].astype(str)

        # Create a new column with a list of all tag names
        all_tags_list = tags['name'].tolist()

        # Filter for "API" tags
        api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()

        if not api_tags.empty:
            # Add metadata columns and all_tags list as a single value (not as a series)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved', 'webLink']:
                api_tags[col] = row.get(col)
            
            # Assign the all_tags list as a single value for the entire set of API tags
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)

            # Append to the filtered DataFrame
            filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)

# Ensure 'lastObserved' is datetime
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Ensure necessary columns exist
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [col for col in required_columns if col not in observed_data_df.columns]

if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# Clean the 'name' column by removing ' Splunk API'
filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)

# Initialize the observed_date column with NaN
filtered_tags['observed_date'] = None

# Iterate through each row and search for matching indicator and opdiv
for index, row in filtered_tags.iterrows():
    # Extract summary and cleaned_name
    summary = row['summary']
    cleaned_name = row['cleaned_name']
    
    # Search for matching rows in the observed data
    match = observed_data_df[(observed_data_df['indicator'] == summary) & (observed_data_df['OpDiv'] == cleaned_name)]
    
    # If a match is found, extract the obs_date
    if not match.empty:
        # Assign the first matching obs_date
        filtered_tags.at[index, 'observed_date'] = match['obs_date'].iloc[0]

filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')

# Drop the 'cleaned_name' column as it is no longer needed
filtered_tags.drop(columns=['cleaned_name'], inplace=True)

# Filter to last 48 hours
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)].copy()

# Make `cutoff` a naive datetime to match the naive `observed_date`
cutoff_naive = cutoff.tz_convert(None)

# Apply the filter directly
recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=1)].copy()
# Extract partner names and remove ' Splunk API' suffix
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

# Group partners per summary
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))]).reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

# Merge partner information back into recent_tags
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

# Filter by dateAdded in the last 30 days
#recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]

# Drop duplicates based on 'summary'
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

# Drop unnecessary columns
columns_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name'
]
recent_tags = recent_tags.drop(columns=[col for col in columns_to_drop if col in recent_tags.columns], errors='ignore')


# Remove rows where 'all_tags' contains 'I&W'
#recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'I&W' in x)]
#recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'htoc_wl' in x)]


recent_tags


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners
0,35760,RITM0589984,2025-06-17T11:58:26Z,45.141.215.88,21,Address,2025-06-16 17:42:26+00:00,2025-06-17T11:58:47Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
3,35760,RITM0589984,2025-06-17T11:58:26Z,192.42.116.215,68,Address,2025-06-16 17:42:28+00:00,2025-06-17T11:58:32Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
6,35760,RITM0589984,2025-06-17T11:58:26Z,192.42.116.178,68,Address,2025-06-16 17:42:12+00:00,2025-06-17T11:58:32Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
9,23630,RITM0588707 / TASK0886419,2025-06-17T09:12:21Z,161.35.53.231,110,Address,2025-06-11 14:51:45+00:00,2025-06-17T11:58:28Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[CMS Splunk API, NIH Splunk API, IHS Splunk AP...",2025-06-17,2,"IHS, NIH"
11,471298,RITM0589984,2025-06-17T11:14:12Z,64.62.197.151,1196,Address,2025-06-16 17:35:09+00:00,2025-06-17T11:58:15Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,3,"CMS, DHA, HRSA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
868,35057,RITM0588707 / TASK0886419,2025-06-17T11:58:15Z,121.196.237.27,52492,Address,2025-06-11 14:46:33+00:00,2025-06-12T15:50:38Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-17,3,"CMS, FDA, HRSA"
871,471298,RITM0588707 / TASK0886419,2025-06-17T11:14:12Z,142.93.115.5,21486,Address,2025-06-11 14:46:22+00:00,2025-06-12T12:42:43Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,6,"CMS, DHA, FDA, HRSA, IHS, OS"
877,471298,RITM0581772/TASK0877779,2025-06-17T11:14:12Z,196.251.72.29,3071360,Address,2025-05-19 11:39:38+00:00,2025-06-12T12:41:59Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,8,"CDC, CMS, DHA, FDA, HRSA, IHS, NIH, OS"
885,471298,NaN,2025-06-17T11:14:12Z,103.108.231.51,36,Address,2025-04-23 15:01:06+00:00,2025-06-12T11:24:51Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, UNC5537, Observed]",2025-06-17,1,DHA


In [16]:
from IPython.display import display

# Get all unique partners (split by comma and flatten)
all_partners = set(
    p.strip()
    for partners in recent_tags['partners'].dropna().unique()
    for p in partners.split(',')
)

# Build buckets: each partner gets all rows where it appears in the partners column
partner_buckets = {
    partner: recent_tags[recent_tags['partners'].str.contains(rf'\b{partner}\b', na=False)]
    for partner in all_partners
}

# Show each partner's dataframe as a table
for partner, df in partner_buckets.items():
    print(f"Partner: {partner} ({len(df)} records)")
    display(df)

Partner: FDA (105 records)


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners
42,35057,INC9102471,2025-06-17T11:58:15Z,123.58.209.112,49746,Address,2025-06-13 14:11:05+00:00,2025-06-17T11:57:57Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,2,"CMS, FDA"
115,471298,RITM0580258/TASK0875884,2025-06-17T11:14:12Z,193.163.125.47,587766,Address,2025-05-14 12:23:46+00:00,2025-06-17T11:38:09Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-17,5,"CMS, DHA, FDA, HRSA, OS"
127,35057,RITM0582833,2025-06-17T11:58:15Z,64.62.197.39,318599,Address,2025-05-21 13:34:40+00:00,2025-06-17T11:38:06Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-17,3,"CMS, FDA, IHS"
133,35057,INC9045771,2025-06-17T11:58:15Z,157.230.221.156,439334,Address,2025-05-07 12:51:43+00:00,2025-06-17T11:38:05Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-17,2,"CMS, FDA"
135,35057,INC9067814,2025-06-17T11:58:15Z,101.89.174.236,106030,Address,2025-05-21 18:39:43+00:00,2025-06-17T11:38:05Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,3,"CMS, FDA, HRSA"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
860,471298,RITM0588707 / TASK0886419,2025-06-17T11:14:12Z,178.128.84.112,15501,Address,2025-06-11 14:46:30+00:00,2025-06-12T23:57:53Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,5,"DHA, FDA, HRSA, IHS, OS"
865,35760,RITM0588707 / TASK0886419,2025-06-17T11:58:26Z,78.153.140.151,3392,Address,2025-06-11 14:46:24+00:00,2025-06-12T17:45:37Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, NIH Splunk API...",2025-06-17,3,"FDA, IHS, OS"
868,35057,RITM0588707 / TASK0886419,2025-06-17T11:58:15Z,121.196.237.27,52492,Address,2025-06-11 14:46:33+00:00,2025-06-12T15:50:38Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-17,3,"CMS, FDA, HRSA"
871,471298,RITM0588707 / TASK0886419,2025-06-17T11:14:12Z,142.93.115.5,21486,Address,2025-06-11 14:46:22+00:00,2025-06-12T12:42:43Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,6,"CMS, DHA, FDA, HRSA, IHS, OS"


Partner: CMS (195 records)


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners
0,35760,RITM0589984,2025-06-17T11:58:26Z,45.141.215.88,21,Address,2025-06-16 17:42:26+00:00,2025-06-17T11:58:47Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
3,35760,RITM0589984,2025-06-17T11:58:26Z,192.42.116.215,68,Address,2025-06-16 17:42:28+00:00,2025-06-17T11:58:32Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
6,35760,RITM0589984,2025-06-17T11:58:26Z,192.42.116.178,68,Address,2025-06-16 17:42:12+00:00,2025-06-17T11:58:32Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
11,471298,RITM0589984,2025-06-17T11:14:12Z,64.62.197.151,1196,Address,2025-06-16 17:35:09+00:00,2025-06-17T11:58:15Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,3,"CMS, DHA, HRSA"
14,471298,RITM0589984,2025-06-17T11:14:12Z,64.62.197.71,1054,Address,2025-06-16 17:35:08+00:00,2025-06-17T11:58:15Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, FDA Splunk API, CMS Splunk AP...",2025-06-17,4,"CMS, DHA, HRSA, IHS"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
848,471298,RITM0588707 / TASK0886419,2025-06-17T11:14:12Z,193.163.125.78,61832,Address,2025-06-11 14:46:28+00:00,2025-06-13T04:14:23Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,6,"CMS, DHA, FDA, HRSA, IHS, OS"
854,35760,RITM0588707 / TASK0886419,2025-06-17T11:58:26Z,193.163.125.217,62228,Address,2025-06-11 14:46:27+00:00,2025-06-13T02:22:48Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-17,6,"CMS, FDA, HRSA, IHS, NIH, OS"
868,35057,RITM0588707 / TASK0886419,2025-06-17T11:58:15Z,121.196.237.27,52492,Address,2025-06-11 14:46:33+00:00,2025-06-12T15:50:38Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-17,3,"CMS, FDA, HRSA"
871,471298,RITM0588707 / TASK0886419,2025-06-17T11:14:12Z,142.93.115.5,21486,Address,2025-06-11 14:46:22+00:00,2025-06-12T12:42:43Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,6,"CMS, DHA, FDA, HRSA, IHS, OS"


Partner: IHS (96 records)


...

,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners
9,23630,RITM0588707 / TASK0886419,2025-06-17T09:12:21Z,161.35.53.231,110,Address,2025-06-11 14:51:45+00:00,2025-06-17T11:58:28Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[CMS Splunk API, NIH Splunk API, IHS Splunk AP...",2025-06-17,2,"IHS, NIH"
52,35760,RITM0589984,2025-06-17T11:58:26Z,172.233.190.104,590375,Address,2025-06-16 17:35:08+00:00,2025-06-17T11:57:21Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-17,5,"CMS, HRSA, IHS, NIH, OS"
103,471298,Executive Summary: \n\nThe following DeepSeek...,2025-06-17T11:14:12Z,www.deepseek.com,6074,Host,2025-01-29 16:27:10+00:00,2025-06-17T11:38:25Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, DeepSeek, OS Splunk API, VA O...",2025-06-17,2,"DHA, NIH"
105,23630,The Department of Veterans Affairs (VA) receiv...,2025-06-17T09:12:21Z,www.sthda.com,10713,Host,2024-04-16 17:33:15+00:00,2025-06-17T11:38:23Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-17,1,NIH
107,471298,The following list of urls and domains are cur...,2025-06-17T11:14:12Z,chat.deepseek.com,12337,Host,2025-01-29 16:27:10+00:00,2025-06-17T11:38:21Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, DeepSeek, OS Splunk API, VA O...",2025-06-17,2,"DHA, NIH"
109,23630,The correlation search is based on an automate...,2025-06-17T09:12:21Z,vtext.com,10291,Host,2024-12-16 18:58:23+00:00,2025-06-17T11:38:20Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Phishing, VA OIS CSOC CTS Spl...",2025-06-17,1,NIH
124,35760,The following indicators were reported in Octo...,2025-06-17T11:58:26Z,207.244.71.82,47177,Address,2023-10-23 19:06:15+00:00,2025-06-17T11:38:08Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Session Tokens, Breach, stolen credentials, S...",2025-06-17,2,"NIH, OS"
170,471298,INC9047855,2025-06-17T11:14:12Z,66.132.159.246,3745326,Address,2025-03-10 14:21:12+00:00,2025-06-17T11:37:59Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning, OS Splunk AP...",2025-06-17,4,"CMS, DHA, NIH, OS"
174,23630,The following indicators were reported in Octo...,2025-06-17T09:12:21Z,207.244.71.84,44536,Address,2023-10-23 19:06:15+00:00,2025-06-17T11:37:59Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Session Tokens, Breach, stolen credentials, S...",2025-06-17,1,NIH
197,471298,CMS received a Volumetrics alert regarding mul...,2025-06-17T11:14:12Z,91.196.152.40,3332903,Address,2025-03-14 11:55:19+00:00,2025-06-17T11:37:55Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-06-17,6,"CMS, DHA, FDA, HRSA, NIH, OS"


Partner: HRSA (128 records)


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners
0,35760,RITM0589984,2025-06-17T11:58:26Z,45.141.215.88,21,Address,2025-06-16 17:42:26+00:00,2025-06-17T11:58:47Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
3,35760,RITM0589984,2025-06-17T11:58:26Z,192.42.116.215,68,Address,2025-06-16 17:42:28+00:00,2025-06-17T11:58:32Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
6,35760,RITM0589984,2025-06-17T11:58:26Z,192.42.116.178,68,Address,2025-06-16 17:42:12+00:00,2025-06-17T11:58:32Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
11,471298,RITM0589984,2025-06-17T11:14:12Z,64.62.197.151,1196,Address,2025-06-16 17:35:09+00:00,2025-06-17T11:58:15Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,3,"CMS, DHA, HRSA"
14,471298,RITM0589984,2025-06-17T11:14:12Z,64.62.197.71,1054,Address,2025-06-16 17:35:08+00:00,2025-06-17T11:58:15Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, FDA Splunk API, CMS Splunk AP...",2025-06-17,4,"CMS, DHA, HRSA, IHS"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
854,35760,RITM0588707 / TASK0886419,2025-06-17T11:58:26Z,193.163.125.217,62228,Address,2025-06-11 14:46:27+00:00,2025-06-13T02:22:48Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-17,6,"CMS, FDA, HRSA, IHS, NIH, OS"
860,471298,RITM0588707 / TASK0886419,2025-06-17T11:14:12Z,178.128.84.112,15501,Address,2025-06-11 14:46:30+00:00,2025-06-12T23:57:53Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,5,"DHA, FDA, HRSA, IHS, OS"
868,35057,RITM0588707 / TASK0886419,2025-06-17T11:58:15Z,121.196.237.27,52492,Address,2025-06-11 14:46:33+00:00,2025-06-12T15:50:38Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-17,3,"CMS, FDA, HRSA"
871,471298,RITM0588707 / TASK0886419,2025-06-17T11:14:12Z,142.93.115.5,21486,Address,2025-06-11 14:46:22+00:00,2025-06-12T12:42:43Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,6,"CMS, DHA, FDA, HRSA, IHS, OS"


Partner: OS (129 records)


,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners
0,35760,RITM0589984,2025-06-17T11:58:26Z,45.141.215.88,21,Address,2025-06-16 17:42:26+00:00,2025-06-17T11:58:47Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
3,35760,RITM0589984,2025-06-17T11:58:26Z,192.42.116.215,68,Address,2025-06-16 17:42:28+00:00,2025-06-17T11:58:32Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
6,35760,RITM0589984,2025-06-17T11:58:26Z,192.42.116.178,68,Address,2025-06-16 17:42:12+00:00,2025-06-17T11:58:32Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-17,3,"CMS, HRSA, OS"
22,471298,RITM0589984,2025-06-17T11:14:12Z,155.94.155.19,1742,Address,2025-06-16 17:35:05+00:00,2025-06-17T11:58:12Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,5,"CMS, DHA, HRSA, IHS, OS"
27,471298,RITM0589984,2025-06-17T11:14:12Z,170.64.166.144,1763,Address,2025-06-16 17:42:37+00:00,2025-06-17T11:58:12Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,4,"DHA, HRSA, IHS, OS"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
854,35760,RITM0588707 / TASK0886419,2025-06-17T11:58:26Z,193.163.125.217,62228,Address,2025-06-11 14:46:27+00:00,2025-06-13T02:22:48Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-17,6,"CMS, FDA, HRSA, IHS, NIH, OS"
860,471298,RITM0588707 / TASK0886419,2025-06-17T11:14:12Z,178.128.84.112,15501,Address,2025-06-11 14:46:30+00:00,2025-06-12T23:57:53Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,5,"DHA, FDA, HRSA, IHS, OS"
865,35760,RITM0588707 / TASK0886419,2025-06-17T11:58:26Z,78.153.140.151,3392,Address,2025-06-11 14:46:24+00:00,2025-06-12T17:45:37Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, NIH Splunk API...",2025-06-17,3,"FDA, IHS, OS"
871,471298,RITM0588707 / TASK0886419,2025-06-17T11:14:12Z,142.93.115.5,21486,Address,2025-06-11 14:46:22+00:00,2025-06-12T12:42:43Z,2025-06-17 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-17,6,"CMS, DHA, FDA, HRSA, IHS, OS"


In [ ]:
import os
import re
import time
import win32com.client as win32

Tippers_Path = r'Z:\HTOC\HTOC Reports\Tippers'

# Start Outlook
outlook = win32.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")
drafts_folder = namespace.GetDefaultFolder(16)  # olFolderDrafts = 16

for partner, df in partner_buckets.items():
    try:
        safe_partner = re.sub(r'[^a-zA-Z0-9_-]', '_', partner)
        partner_folder = os.path.join(Tippers_Path, safe_partner)
        os.makedirs(partner_folder, exist_ok=True)

        # Save CSV to disk (Outlook needs a real file to attach)
        csv_filename = f"{safe_partner}_tippers.csv"
        csv_path = os.path.join(partner_folder, csv_filename)
        df.to_csv(csv_path, index=False)
        time.sleep(0.5)  # ensure write completes

        # Create mail item and save to Drafts
        mail = outlook.CreateItem(0)
        mail.Subject = f"Tippers Data for {partner}"
        mail.To = f"{safe_partner.lower()}@yourdomain.com"
        mail.Body = f"Attached is the Tippers CSV file for {partner}."
        mail.Attachments.Add(Source=csv_path)
        
        # Move to Drafts folder
        mail = mail.Move(drafts_folder)
        print(f"Draft saved in Outlook for: {partner}")

    except Exception as e:
        print(f"Error processing partner '{partner}': {e}")


Draft saved in Outlook for: FDA
Draft saved in Outlook for: CMS
Draft saved in Outlook for: IHS
Draft saved in Outlook for: CDC
Draft saved in Outlook for: HHS
Draft saved in Outlook for: DHA
Draft saved in Outlook for: NIH
Draft saved in Outlook for: HRSA
Draft saved in Outlook for: OS


In [ ]:
import os
import re
from email.message import EmailMessage
from email.utils import formatdate
from email.policy import SMTP

Tippers_Path = r'Z:\HTOC\HTOC Reports\Tippers'
os.makedirs(Tippers_Path, exist_ok=True)

for partner, df in partner_buckets.items():
    try:
        safe_partner = re.sub(r'[^a-zA-Z0-9_-]', '_', partner)
        partner_folder = os.path.join(Tippers_Path, safe_partner)
        os.makedirs(partner_folder, exist_ok=True)

        # Prepare CSV attachment (in memory)
        csv_bytes = df.to_csv(index=False).encode('utf-8')
        csv_filename = f"{safe_partner}_tippers.csv"

        # Create proper email message
        msg = EmailMessage(policy=SMTP)
        msg['Subject'] = f"Tippers Data for {partner}"
        msg['From'] = "jaytlin.askew@hhs.gov"
        msg['To'] = f"jaytlin.askew@hhs.gov"
        msg['Date'] = formatdate(localtime=True)
        msg.set_content(f"Attached is the Tippers CSV file for {partner}.")

        # Attach CSV as MIME part
        msg.add_attachment(csv_bytes, maintype='text', subtype='csv', filename=csv_filename)

        # Save to .eml file
        eml_path = os.path.join(partner_folder, f"{safe_partner}_tippers.eml")
        with open(eml_path, 'wb') as f:
            f.write(bytes(msg))

        print(f".eml file saved: {eml_path}")

    except Exception as e:
        print(f"Error processing partner '{partner}': {e}")
        continue


In [6]:
import pandas as pd
import requests
import time
import ipaddress
from datetime import datetime
import concurrent.futures

# API Keys
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"

# Headers for API requests
VT_HEADERS = {"x-apikey": VT_API_KEY}

# API URLs
VT_URL_IP = "https://www.virustotal.com/api/v3/ip_addresses/{}"
VT_URL_DOMAIN = "https://www.virustotal.com/api/v3/domains/{}"

def is_ip(value):
    """ Check if the given value is a valid IP address. """
    try:
        ipaddress.ip_address(value)
        return True
    except ValueError:
        return False

def determine_query_type(query):
    """ Determine if the query is an IP, domain, or hostname. """
    if is_ip(query):
        return "ip"
    elif "." in query:
        return "hostname"
    else:
        return "domain"

def fetch_virustotal_data(query):
    """Fetch data from VirusTotal API for IP or Domain."""
    query_type = determine_query_type(query)
    url = VT_URL_IP.format(query) if query_type == "ip" else VT_URL_DOMAIN.format(query)

    try:
        response = requests.get(url, headers=VT_HEADERS, verify=False)
        if response.status_code == 429:
            print("Rate limit hit. Sleeping for 60 seconds...")
            time.sleep(60)
            response = requests.get(url, headers=VT_HEADERS, verify=False)
        response.raise_for_status()
        data = response.json().get("data", {}).get("attributes", {})

        return {
            "search_term": query,
            "asn": data.get('asn'),
            "as_owner": data.get('as_owner'),
            "country": data.get('country'),
            "network": data.get('network'),
            "last_analysis_stats": data.get('last_analysis_stats'),
            "reputation": data.get('reputation'),
            "total_votes": data.get('total_votes'),
            "open_ports": [s.get("port") for s in data.get("services", []) if "port" in s],
            "link": f"https://www.virustotal.com/gui/ip-address/{query}" if query_type == "ip" else f"https://www.virustotal.com/gui/domain/{query}"
        }

    except Exception as e:
        print(f"VirusTotal Error for {query}: {e}")
        return None

def process_indicator(indicator, observed_by, partner_count):
    """Fetch data for a single indicator."""
    timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    # Only one thread at a time to avoid hitting the rate limit
    vt_data = fetch_virustotal_data(indicator)

    if vt_data:
        vt_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    return vt_data

def main(recent_tags):
    """ Main function to process indicators with VT API rate limits. """
    if 'summary' not in recent_tags.columns:
        print("The 'summary' column is missing.")
        return pd.DataFrame(), pd.DataFrame()

    search_terms = recent_tags['summary'].dropna().unique().tolist()
    print(f"Processing {len(search_terms)} unique search terms.")

    vt_results = []
    requests_made = 0
    max_requests_per_day = 500
    max_requests_per_minute = 4
    minute_window = 60  # seconds
    minute_batch = []

    for i, indicator in enumerate(search_terms):
        if requests_made >= max_requests_per_day:
            print("Daily VirusTotal API request limit reached (500). Stopping.")
            break

        partners = recent_tags.loc[recent_tags['summary'] == indicator, 'partners'].values
        partner_count = recent_tags.loc[recent_tags['summary'] == indicator, 'partner_count'].values
        observed_by = partners[0] if len(partners) > 0 else "N/A"
        partner_count = partner_count[0] if len(partner_count) > 0 else "N/A"

        vt_data = process_indicator(indicator, observed_by, partner_count)
        if vt_data:
            vt_results.append(vt_data)
        requests_made += 1

        # Throttle to 4 requests per minute
        minute_batch.append(time.time())
        if len(minute_batch) == max_requests_per_minute:
            elapsed = time.time() - minute_batch[0]
            if elapsed < minute_window:
                sleep_time = minute_window - elapsed
                print(f"VirusTotal minute rate limit reached (4/min). Sleeping for {int(sleep_time)} seconds...")
                time.sleep(sleep_time)
            minute_batch = []

    vt_df = pd.DataFrame(vt_results)
    return vt_df


if __name__ == "__main__":
    # Limit to first 50 rows for testing
    test_tags = recent_tags.head(50).copy()
    vt_df = main(test_tags)
    print("Script completed successfully.")

Processing 50 unique search terms.
Rate limit hit. Sleeping for 60 seconds...
VirusTotal Error for 45.141.215.88: 429 Client Error: Too Many Requests for url: https://www.virustotal.com/api/v3/ip_addresses/45.141.215.88
Rate limit hit. Sleeping for 60 seconds...


: 

: 

In [ ]:
vt_df

(Empty DataFrame
 Columns: []
 Index: [],)

In [ ]:
# Unnest the 'last_analysis_stats' dictionary into separate columns in vt_df
if 'last_analysis_stats' in vt_df.columns:
    stats_df = pd.json_normalize(vt_df['last_analysis_stats'])
    stats_df.columns = [f'last_analysis_{col}' for col in stats_df.columns]
    vt_df = pd.concat([vt_df.drop(columns=['last_analysis_stats']), stats_df], axis=1)

vt_df

,search_term,asn,as_owner,country,network,reputation,total_votes,open_ports,link,timestamp,observed_by,partner_count,last_analysis_malicious,last_analysis_suspicious,last_analysis_undetected,last_analysis_harmless,last_analysis_timeout
0,142.93.13.29,14061.0,DIGITALOCEAN-ASN,US,142.93.0.0/16,-2,"{'harmless': 0, 'malicious': 2}",[],https://www.virustotal.com/gui/ip-address/142....,2025-06-04 12:09:34,"CMS, DHA, FDA, HRSA, IHS, OS",6,16,4,25,49,0
1,uploads-ssl.webflow.com,NaN,None,None,None,0,"{'harmless': 0, 'malicious': 0}",[],https://www.virustotal.com/gui/domain/uploads-...,2025-06-04 12:09:34,"CMS, DHA, HRSA, IHS, NIH",5,0,0,29,65,0
2,91.196.152.44,213412.0,ONYPHE SAS,FR,91.196.152.0/24,0,"{'harmless': 0, 'malicious': 0}",[],https://www.virustotal.com/gui/ip-address/91.1...,2025-06-04 12:09:34,"CMS, DHA, FDA, HRSA, IHS, OS",6,9,3,27,55,0
3,196.251.70.76,401120.0,CHEAPY-HOST,SC,196.251.70.0/23,0,"{'harmless': 0, 'malicious': 0}",[],https://www.virustotal.com/gui/ip-address/196....,2025-06-04 12:09:34,"CDC, CMS, DHA, FDA, IHS, NIH, OS",7,6,2,29,57,0
4,ctrk.klclick2.com,NaN,None,None,None,0,"{'harmless': 0, 'malicious': 0}",[],https://www.virustotal.com/gui/domain/ctrk.klc...,2025-06-04 12:09:34,"CMS, DHA, HRSA, IHS, NIH",5,0,0,30,64,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,118.193.72.187,135377.0,UCLOUD INFORMATION TECHNOLOGY HK LIMITED,PH,118.193.64.0/20,-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/118....,2025-06-04 12:10:07,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",7,10,2,30,52,0
166,196.251.70.87,401120.0,CHEAPY-HOST,SC,196.251.70.0/23,-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/196....,2025-06-04 12:10:07,"CDC, CMS, DHA, FDA, HRSA, IHS, NIH, OS",8,10,3,27,54,0
167,198.55.98.76,214940.0,Kprohost LLC,US,198.55.98.0/24,-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/198....,2025-06-04 12:10:06,"CDC, CMS, DHA, FDA, HRSA, IHS, NIH, OS",8,9,3,28,54,0
168,196.251.73.140,401120.0,CHEAPY-HOST,SC,196.251.72.0/23,0,"{'harmless': 0, 'malicious': 0}",[],https://www.virustotal.com/gui/ip-address/196....,2025-06-04 12:10:08,"CMS, DHA, FDA, IHS, NIH, OS",6,6,3,29,56,0


In [ ]:
# Merge 'last_analysis_malicious' from vt_df into recent_tags based on summary/search_term
recent_tags = recent_tags.merge(
    vt_df[['search_term', 'last_analysis_malicious']],
    left_on='summary',
    right_on='search_term',
    how='left'
).drop(columns=['search_term'])

recent_tags

,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,last_analysis_malicious
0,471298,2025-06-04T09:35:13Z,TASK0882701 / RITM0585661,193.174.89.19,26245,Address,2025-06-02 18:33:32+00:00,2025-06-04T11:38:40Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, CMS Splunk API...",2025-06-04,4,"CMS, DHA, HRSA, IHS",10
1,471298,2025-06-04T09:35:13Z,TASK0882701 / RITM0585661,196.251.70.216,34999,Address,2025-06-02 18:33:46+00:00,2025-06-04T08:02:56Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-04,7,"CDC, CMS, DHA, FDA, IHS, NIH, OS",10
2,471298,2025-06-04T09:35:13Z,TASK0882701 / RITM0585661,118.26.111.94,80021,Address,2025-06-02 19:00:24+00:00,2025-06-04T08:02:50Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",8
3,471298,2025-06-04T09:35:13Z,Domain associated with Legion Loader and mali...,uploads-ssl.webflow.com,38363,Host,2025-04-20 17:39:55+00:00,2025-06-04T07:38:01Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Fake Captcha, malware dropper...",2025-06-04,5,"CMS, DHA, HRSA, IHS, NIH",0
4,471298,2025-06-04T09:35:13Z,NaN,ctrk.klclick2.com,58465,Host,2025-02-28 16:38:16+00:00,2025-06-04T07:37:56Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-04,5,"CMS, DHA, HRSA, IHS, NIH",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,471298,2025-06-04T09:35:13Z,RITM0582833,83.222.190.66,46893848,Address,2025-05-21 13:34:38+00:00,2025-06-02T06:12:35Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-04,6,"CMS, DHA, FDA, HRSA, IHS, OS",10
166,471298,2025-06-04T09:35:13Z,CMS received volumetrics alerts about a set of...,83.222.191.6,1081044,Address,2025-05-23 12:00:30+00:00,2025-06-02T06:12:34Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning, OS Splunk AP...",2025-06-04,5,"CMS, DHA, HRSA, NIH, OS",10
167,471298,2025-06-04T09:35:13Z,RITM0581780/TASK0877793,85.208.104.4,400175,Address,2025-05-19 11:39:38+00:00,2025-06-02T06:12:34Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",4
168,471298,2025-06-04T09:35:13Z,RITM0581772/TASK0877779,180.214.238.62,439998,Address,2025-05-19 11:39:46+00:00,2025-06-02T06:12:33Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-03,5,"CMS, DHA, FDA, HRSA, OS",7


In [ ]:
malicious = recent_tags[recent_tags['last_analysis_malicious'] == 0]

malicious

,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,last_analysis_malicious
3,471298,2025-06-04T09:35:13Z,Domain associated with Legion Loader and mali...,uploads-ssl.webflow.com,38363,Host,2025-04-20 17:39:55+00:00,2025-06-04T07:38:01Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Fake Captcha, malware dropper...",2025-06-04,5,"CMS, DHA, HRSA, IHS, NIH",0
4,471298,2025-06-04T09:35:13Z,NaN,ctrk.klclick2.com,58465,Host,2025-02-28 16:38:16+00:00,2025-06-04T07:37:56Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-04,5,"CMS, DHA, HRSA, IHS, NIH",0
55,30479,2025-06-04T09:04:59Z,IB-23-10028 Towing Company Email used to Deliv...,185.230.63.171,197909,Address,2021-04-27 11:57:47+00:00,2025-06-04T07:37:38Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, HHS Splunk Production API, Ph...",2025-06-03,4,"CDC, CMS, HHS, NIH",0
102,471298,2025-06-04T09:35:13Z,TASK0882699 / RITM0585658,34.83.195.219,120829,Address,2025-06-02 19:05:35+00:00,2025-06-03T15:37:48Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-03,7,"CMS, DHA, FDA, HRSA, IHS, NIH, OS",0
109,471298,2025-06-04T09:35:13Z,TASK0882701 / RITM0585661,35.157.198.100,10023,Address,2025-06-02 18:33:33+00:00,2025-06-03T12:36:29Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, CMS Splunk API, HRSA Splunk A...",2025-06-03,5,"CMS, DHA, HRSA, IHS, NIH",0
127,471298,2025-06-04T09:35:13Z,The following list of urls and domains are cur...,chat.deepseek.com,9228,Host,2025-01-29 16:27:10+00:00,2025-06-03T07:33:16Z,2025-06-04 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, DeepSeek, OS Splunk API, VA O...",2025-06-04,4,"CMS, DHA, HRSA, NIH",0
